## 1. Import Libraries & Load Raw Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, classification_report


np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
TARGET_COLUMN = "GamePopularity"



In [ ]:
df = pd.read_csv('Data/classification/train_data.csv')
print(f"Raw dataset shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print(f"\n--- Data Types ---")
print(df.dtypes.value_counts())

print(f"\n--- First few rows ---")
print(df.head())

Raw dataset shape: (11357, 78)
Total rows: 11357
Total columns: 78

--- Data Types ---
bool       35
object     24
int64      17
float64     2
Name: count, dtype: int64

--- First few rows ---
   QueryID  ResponseID                  QueryName               ResponseName  \
0       10          10             Counter-Strike             Counter-Strike   
1       20          20      Team Fortress Classic      Team Fortress Classic   
2       30          30              Day of Defeat              Day of Defeat   
3       40          40         Deathmatch Classic         Deathmatch Classic   
4       50          50  Half-Life: Opposing Force  Half-Life: Opposing Force   

  ReleaseDate  RequiredAge  DemoCount  DeveloperCount  DLCCount  Metacritic  \
0  Nov 1 2000            0          0               1         0          88   
1  Apr 1 1999            0          0               1         0           0   
2  May 1 2003            0          0               1         0          79   
3  Jun 1 2

## Data Engineering & Feature Creation

In [ ]:
df_processed = df.copy()

text_cols = ['ShortDescrip', 'DetailedDescrip', 'AboutText']
text_cols = [col for col in text_cols if col in df_processed.columns]

if text_cols:
    for col in text_cols:
        df_processed[f'{col}_length'] = df_processed[col].fillna('').str.len()
        df_processed[f'{col}_word_count'] = df_processed[col].fillna('').str.split().str.len()
    
    combined_text = df_processed[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
    df_processed['total_text_length'] = combined_text.str.len()
    df_processed['total_word_count'] = combined_text.str.split().str.len()
    df_processed['has_website'] = (df_processed['Website'].fillna('') != '').astype(int)
    
    df_processed = df_processed.drop(columns=text_cols)
    print(f"✓ Created text-based features")

print(f"✓ Creating interaction features...")
interaction_pairs = [
    ('SteamSpyOwners', 'SteamSpyPlayersEstimate'),
    ('PriceFinal', 'Metacritic'),
    ('DLCCount', 'AchievementCount'),
    ('DeveloperCount', 'PublisherCount'),
]

for feat1, feat2 in interaction_pairs:
    if feat1 in df_processed.columns and feat2 in df_processed.columns:
        df_processed[f'{feat1}_x_{feat2}'] = df_processed[feat1] * df_processed[feat2]
        df_processed[f'{feat1}_div_{feat2}'] = np.where(
            df_processed[feat2] != 0, 
            np.log1p(df_processed[feat1]) / (np.log1p(df_processed[feat2]) + 1e-6),
            0
        )

category_cols = [col for col in df_processed.columns if col.startswith('Category')]
genre_cols = [col for col in df_processed.columns if col.startswith('Genre')]
platform_cols = [col for col in df_processed.columns if col.startswith('Platform')]
req_cols = [col for col in df_processed.columns if 'Reqs' in col]

df_processed['num_categories'] = df_processed[category_cols].sum(axis=1) if category_cols else 0
df_processed['num_genres'] = df_processed[genre_cols].sum(axis=1) if genre_cols else 0
df_processed['num_platforms'] = df_processed[platform_cols].sum(axis=1) if platform_cols else 0

skewed_features = ['SteamSpyOwners', 'SteamSpyPlayersEstimate', 'ScreenshotCount', 'DLCCount']
for feat in skewed_features:
    if feat in df_processed.columns:
        df_processed[f'{feat}_log'] = np.log1p(df_processed[feat])

id_url_cols = [col for col in df_processed.columns if any(x in col.lower() for x in ['id', 'url', 'email', 'image', 'background', 'header'])]
df_processed = df_processed.drop(columns=id_url_cols, errors='ignore')

print(f"\n--- Feature Engineering Complete ---")
print(f"Original shape: {df.shape}")
print(f"Processed shape: {df_processed.shape}")
print(f"New features created: {df_processed.shape[1] - df.shape[1]}")

✓ Created text-based features
✓ Creating interaction features...

--- Feature Engineering Complete ---
Original shape: (11357, 78)
Processed shape: (11357, 93)
New features created: 15


## 4. Data Preprocessing & Preparation

In [ ]:
X = df_processed.drop(columns=[ 'GamePopularity'])
y_raw = df_processed['GamePopularity'].copy()

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y_raw)
print(f"Encoded classes: {list(le_target.classes_)} -> {list(range(len(le_target.classes_)))}")

print(f"Initial shapes - X: {X.shape}, y: {y_encoded.shape}")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

print(f"\nMissing values before imputation: {X.isnull().sum().sum()}")

# if len(numeric_features) > 0:
#     numeric_imputer = SimpleImputer(strategy='median')
#     X[numeric_features] = numeric_imputer.fit_transform(X[numeric_features])

if len(categorical_features) > 0:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    X[categorical_features] = categorical_imputer.fit_transform(X[categorical_features])

    le_dict = {}
    for col in categorical_features:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le

# print(f"Missing values after imputation: {X.isnull().sum().sum()}")

# NO log transform for classification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, shuffle=True
)

print(f"\n--- Data Split ---")
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train target distribution: {np.bincount(y_train)}")
print(f"Test  target distribution: {np.bincount(y_test)}")

Encoded classes: ['High', 'Low', 'Medium'] -> [0, 1, 2]
Initial shapes - X: (11357, 92), y: (11357,)
Numeric features: 41
Categorical features: 16

Missing values before imputation: 2715

--- Data Split ---
Train set: (9085, 92)
Test set: (2272, 92)
Train target distribution: [1026 7534  525]
Test  target distribution: [ 271 1874  127]


In [ ]:

# CELL 1: Random Forest Classifier with pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif  # changed to f_classif for classification
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

K_FEATURES = min(30, X_train.shape[1])

print("=" * 70)
print("TRAINING Random Forest CLASSIFIER")
print("=" * 70)

rf_pipeline = Pipeline([
    ("imputer",            SimpleImputer(strategy="mean")),
    ("variance_threshold", VarianceThreshold(threshold=0.0)),
    ("feature_selection",  SelectKBest(f_classif, k=K_FEATURES)),   # classification scoring
    ("model",              RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

rf_pipeline.fit(X_train, y_train)

rf_pred_train = rf_pipeline.predict(X_train)
rf_pred_test  = rf_pipeline.predict(X_test)

rf_acc_train = accuracy_score(y_train, rf_pred_train)
rf_acc_test  = accuracy_score(y_test, rf_pred_test)
rf_f1_test   = f1_score(y_test, rf_pred_test, average='macro')
rf_cm_test   = confusion_matrix(y_test, rf_pred_test)

print(f"✓ Random Forest trained")
print(f"  Accuracy (Train): {rf_acc_train:.4f}")
print(f"  Accuracy (Test):  {rf_acc_test:.4f}")
print(f"  F1-macro (Test):  {rf_f1_test:.4f}")
print(f"  Classification Report:\n{classification_report(y_test, rf_pred_test, target_names=le_target.classes_)}")

# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(rf_cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

TRAINING Random Forest CLASSIFIER


ValueError: Cannot use mean strategy with non-numeric data:
could not convert string to float: 'Among the Heavens'

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("\n" + "=" * 70)
print("TRAINING SVC")
print("=" * 70)

svc_pipeline = Pipeline([
    ("imputer",            SimpleImputer(strategy="mean")),
    ("variance_threshold", VarianceThreshold(threshold=0.0)),
    ("scaler",             StandardScaler()),
    ("model",              SVC(kernel="rbf", C=10, gamma='scale', probability=False)),
])

param_grid = {
    "model__C":       [1, 10],
    "model__gamma":   ['scale', 0.1],
}

grid_search = GridSearchCV(
    svc_pipeline,
    param_grid,
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    scoring="accuracy",          # classification metric
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

best_svc = grid_search.best_estimator_

svc_pred_train = best_svc.predict(X_train)
svc_pred_test  = best_svc.predict(X_test)

svc_acc_train = accuracy_score(y_train, svc_pred_train)
svc_acc_test  = accuracy_score(y_test, svc_pred_test)
svc_f1_test   = f1_score(y_test, svc_pred_test, average='macro')
svc_cm_test   = confusion_matrix(y_test, svc_pred_test)

print(f"✓ SVC trained")
print(f"  Accuracy (Train): {svc_acc_train:.4f}")
print(f"  Accuracy (Test):  {svc_acc_test:.4f}")
print(f"  F1-macro (Test):  {svc_f1_test:.4f}")
print(f"  Classification Report:\n{classification_report(y_test, svc_pred_test, target_names=le_target.classes_)}")

plt.figure(figsize=(6,5))
sns.heatmap(svc_cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('SVC Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

print("\n" + "=" * 70)
print("TRAINING Logistic Regression (with Polynomial Features)")
print("=" * 70)

poly_pipeline = Pipeline([
    ("imputer",            SimpleImputer(strategy="mean")),
    ("variance_threshold", VarianceThreshold(threshold=0.0)),
    ("feature_selection",  SelectKBest(f_classif, k=K_FEATURES)),
    ("scaler",             StandardScaler()),
    ("poly",               PolynomialFeatures(degree=2, include_bias=True)),
    ("model",              LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42)),
])

poly_pipeline.fit(X_train, y_train)

poly_pred_train = poly_pipeline.predict(X_train)
poly_pred_test  = poly_pipeline.predict(X_test)

poly_acc_train = accuracy_score(y_train, poly_pred_train)
poly_acc_test  = accuracy_score(y_test, poly_pred_test)
poly_f1_test   = f1_score(y_test, poly_pred_test, average='macro')
poly_cm_test   = confusion_matrix(y_test, poly_pred_test)

print(f"✓ Logistic Regression (poly) trained")
print(f"  Accuracy (Train): {poly_acc_train:.4f}")
print(f"  Accuracy (Test):  {poly_acc_test:.4f}")
print(f"  F1-macro (Test):  {poly_f1_test:.4f}")
print(f"  Classification Report:\n{classification_report(y_test, poly_pred_test, target_names=le_target.classes_)}")

plt.figure(figsize=(6,5))
sns.heatmap(poly_cm_test, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Logistic Reg. (Poly) Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier as _RFForRFECV

print("\n" + "=" * 70)
print("TRAINING Logistic Regression (with RFECV)")
print("=" * 70)

lr_pipeline = Pipeline([
    ("imputer",            SimpleImputer(strategy="mean")),
    ("variance_threshold", VarianceThreshold(threshold=0.0)),
    ("feature_selection",  RFECV(
        estimator=_RFForRFECV(n_estimators=80, random_state=42, n_jobs=-1),
        step=5,
        min_features_to_select=20,
        cv=KFold(n_splits=3, shuffle=True, random_state=42),
        scoring="accuracy",          
        n_jobs=1,
    )),
    ("model", LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)

lr_pred_train = lr_pipeline.predict(X_train)
lr_pred_test  = lr_pipeline.predict(X_test)

lr_acc_train = accuracy_score(y_train, lr_pred_train)
lr_acc_test  = accuracy_score(y_test, lr_pred_test)
lr_f1_test   = f1_score(y_test, lr_pred_test, average='macro')
lr_cm_test   = confusion_matrix(y_test, lr_pred_test)

selector    = lr_pipeline.named_steps["feature_selection"]
var_filter  = lr_pipeline.named_steps["variance_threshold"]
cols_after_var   = X_train.columns[var_filter.get_support()]
selected_cols    = cols_after_var[selector.support_]

print(f"✓ Logistic Regression (RFECV) trained")
print(f"  Features selected: {selector.n_features_} / {X_train.shape[1]}")
print(f"  Best CV accuracy: {selector.cv_results_['mean_test_score'].max():.4f}")
print(f"  Accuracy (Train): {lr_acc_train:.4f}")
print(f"  Accuracy (Test):  {lr_acc_test:.4f}")
print(f"  F1-macro (Test):  {lr_f1_test:.4f}")
print(f"  Classification Report:\n{classification_report(y_test, lr_pred_test, target_names=le_target.classes_)}")

plt.figure(figsize=(6,5))
sns.heatmap(lr_cm_test, annot=True, fmt='d', cmap='Purples',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Logistic Reg. (RFECV) Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:

# CELL 5: Decision Tree Classifier (replacing Decision Tree Regressor)
from sklearn.tree import DecisionTreeClassifier

print("\n" + "=" * 70)
print("TRAINING Decision Tree CLASSIFIER")
print("=" * 70)

dtmodel = DecisionTreeClassifier(max_depth=5, random_state=42)
dtmodel.fit(X_train, y_train)

dt_pred_train = dtmodel.predict(X_train)
dt_pred_test  = dtmodel.predict(X_test)

dt_acc_train = accuracy_score(y_train, dt_pred_train)
dt_acc_test  = accuracy_score(y_test, dt_pred_test)
dt_f1_test   = f1_score(y_test, dt_pred_test, average='macro')
dt_cm_test   = confusion_matrix(y_test, dt_pred_test)

print(f"✓ Decision Tree trained")
print(f"  Accuracy (Train): {dt_acc_train:.4f}")
print(f"  Accuracy (Test):  {dt_acc_test:.4f}")
print(f"  F1-macro (Test):  {dt_f1_test:.4f}")
print(f"  Classification Report:\n{classification_report(y_test, dt_pred_test, target_names=le_target.classes_)}")

plt.figure(figsize=(6,5))
sns.heatmap(dt_cm_test, annot=True, fmt='d', cmap='Reds',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()